# CS781 Phase 1 — LegalEncDec · 50.8M parameters · 300M tokens

**Select GPU T4 ×2, enable Internet, enable Kaggle Secret `HF` or `HF_TOKEN`, then Run All.**

A genuine encoder–decoder Transformer trained from scratch: **4 encoder layers + 4 decoder layers**, with cross-attention. It fetches the four training sources, selects and cleans data, trains, saves `best.pt` and resumable `latest.pt`, then runs your unchanged Phase 1 evaluator. Your fixed 50K tokenizer is embedded. No submission step.

**Budget:** 299,958,272 corpus tokens (300M rounded down to full updates). Each training example uses 768 source tokens and 256 continuation targets. This means about 225M source tokens + 75M supervised target tokens; it does **not** mean 300M target tokens. Each prepared block is visited once. BOS/EOS separators count toward the corpus budget.

**10-hour session cap:** training cutoff 8h40, generation 9h15, evaluation 9h40, final backup 9h50. Completion depends on download and measured GPU speed. A shortfall is reported honestly. The notebook measures throughput before training and never shrinks the target to fit a speed estimate.

Uses separate output and private-backup names from the 90M experiment. HF write permission enables private recovery backups; otherwise local checkpoints require the VM's files to survive. No trained weights are included.


## 1. Configuration

In [ ]:
import os, sys, time, json
from pathlib import Path

# Run All defaults — no edits required.
TRAIN_TOKENS = 300_000_000
OUTPUT_DIR = Path('/kaggle/working/cs781_encdec_50m_300m')
REMOTE_BACKUP = True
BACKUP_REPO = None  # automatic private repository under your HF account

SESSION_START = float(os.environ.setdefault('CS781_ENCDEC_START', str(time.time())))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
settings = {
    'train_tokens': TRAIN_TOKENS,
    'remote_backup': REMOTE_BACKUP,
    'backup_repo': BACKUP_REPO,
    'preparation_minutes': 75,
    'session_start': SESSION_START,
    'training_deadline': SESSION_START + 8*3600 + 40*60,
    'generation_deadline': SESSION_START + 9*3600 + 15*60,
    'evaluation_deadline': SESSION_START + 9*3600 + 40*60,
    'final_deadline': SESSION_START + 9*3600 + 50*60,
}
(OUTPUT_DIR/'settings.json').write_text(json.dumps(settings, indent=2))
print('Requested training tokens:', f'{TRAIN_TOKENS:,}')
print('Output:', OUTPUT_DIR)

## 2. Automatic runtime setup
The folded cell writes the training/data/recovery utilities included in this notebook. All source files are available under `OUTPUT_DIR` for inspection. It contains no weights or credentials. Torch runs in fresh worker processes; pip is constrained to retain Kaggle's installed Torch/CUDA packages.

In [ ]:
import base64, gzip, hashlib
SUPPORT_B64 = '...'
support_raw = gzip.decompress(base64.b64decode(SUPPORT_B64))
assert hashlib.sha256(support_raw).hexdigest() == '63f0b338252b175259d33b85b741e8f9314650d54d1af1bbfa4da0d9445a9a15'
for name, content in json.loads(support_raw).items():
    (OUTPUT_DIR/name).write_text(content)
del SUPPORT_B64, support_raw
if str(OUTPUT_DIR) not in sys.path: sys.path.insert(0, str(OUTPUT_DIR))
print('Training and preprocessing utilities ready.')


In [ ]:
import importlib.metadata
from runtime import run

# Keep the Kaggle GPU installation intact.
constraints = []
for package in ['torch', 'torchvision', 'torchaudio', 'numpy']:
    try:
        constraints.append(package + '==' + importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        pass
if not any(x.startswith('torch==') for x in constraints):
    raise RuntimeError('Select Kaggle GPU T4 x2 before running.')
(OUTPUT_DIR/'constraints.txt').write_text('\n'.join(constraints)+'\n')
run([sys.executable, '-m', 'pip', 'install', '-q', '--constraint', OUTPUT_DIR/'constraints.txt',
     'tokenizers==0.21.1', 'huggingface_hub==0.30.2', 'transformers==4.51.3',
     'bert-score==0.3.13', 'ijson==3.3.0', 'requests==2.32.3'],
    OUTPUT_DIR, min(time.time()+15*60, settings['final_deadline']), 'install', grace=5)
os.environ.update(TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='2',
                  HF_HUB_DOWNLOAD_TIMEOUT='30', HF_HUB_ETAG_TIMEOUT='20')
print('Dependencies ready.')

## 3. Hugging Face token

In [ ]:
from kaggle_secrets import UserSecretsClient

if not os.environ.get('HF_TOKEN'):
    secrets = UserSecretsClient()
    for secret_name in ['HF', 'HF_TOKEN']:
        try:
            value = secrets.get_secret(secret_name)
            if value:
                os.environ['HF_TOKEN'] = value
                del value
                break
        except Exception:
            pass
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('Add and enable a Kaggle Secret named HF or HF_TOKEN with competition dataset access.')
print('HF token loaded securely. No login prompt is required.')

## 4. Load your tokenizer and evaluator
Both attached files are embedded byte-for-byte and checksum verified. You do not need to attach them to Kaggle again. Expand these cells to inspect the payload/evaluator.

In [ ]:
import base64, gzip, hashlib
TOKENIZER_B64 = '...'
raw = gzip.decompress(base64.b64decode(TOKENIZER_B64))
assert hashlib.sha256(raw).hexdigest() == '636cf69e301bdd6ed22f1d89cb9ce01edb4e104f9456e5b99c6fa30fcd882233'
(OUTPUT_DIR/'tokenizer.json').write_bytes(raw)
del TOKENIZER_B64, raw
print('Original tokenizer restored and checksum verified.')

In [ ]:
EVALUATOR_SOURCE = '#!/usr/bin/env python3\n"""Local evaluator for Phase 1.\n\nUsage:\n  python evaluate_phase1.py --truth truth.json --prediction prediction.json\n\nRequires: bert-score and torch for the official primary metric.\n"""\nimport argparse, json, math, sys\nfrom collections import Counter\n\ntry:\n    import bert_score\nexcept ImportError as exc:\n    raise SystemExit("Missing dependency: bert-score. Install it from requirements.docx / requirements.txt.") from exc\n\nclass DuplicateJSONKeyError(ValueError): pass\n\ndef reject_constant(v): raise ValueError(f"Non-standard JSON numeric value \'{v}\' is not allowed.")\n\ndef pairs_no_duplicates(pairs):\n    out = {}; dup = []\n    for k, v in pairs:\n        if k in out: dup.append(k)\n        out[k] = v\n    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")\n    return out\n\ndef load_json(path):\n    try:\n        with open(path, encoding=\'utf-8\') as f:\n            return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)\n    except json.JSONDecodeError as e:\n        raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e\n\ndef validate(data, name):\n    if not isinstance(data, dict) or not data: raise ValueError(f"{name} must be a non-empty JSON object mapping IDs to records.")\n    for i, rec in data.items():\n        if not isinstance(i, str) or not i.strip() or not isinstance(rec, dict) or \'text\' not in rec or not isinstance(rec[\'text\'], str) or not rec[\'text\'].strip():\n            raise ValueError(f"Invalid record for ID {i!r} in {name}; each record must contain non-empty string field \'text\'.")\n\ndef lcs_length(a,b):\n    if not a or not b: return 0\n    if len(a) < len(b): a,b=b,a\n    masks={}\n    for j,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<j)\n    state=0\n    for t in b:\n        m=masks.get(t,0); x=m|state; y=(state<<1)|1; state=x & ~(x-y)\n    return state.bit_count() if hasattr(int,\'bit_count\') else bin(state).count(\'1\')\n\ndef rouge_l_f1(p,r):\n    pt=p.split(); rt=r.split()\n    if not pt or not rt: return 0.0\n    l=lcs_length(pt,rt)\n    if not l: return 0.0\n    pp=l/len(pt); rr=l/len(rt)\n    return 0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)\n\ndef ngrams(t,n): return Counter(tuple(t[i:i+n]) for i in range(len(t)-n+1)) if len(t)>=n else Counter()\n\ndef bleu4(preds, refs):\n    order=4; clipped=[0]*order; totals=[0]*order; pl=rl=0\n    for p,r in zip(preds,refs):\n        pl+=len(p); rl+=len(r)\n        for n in range(1,order+1):\n            pn=ngrams(p,n); rn=ngrams(r,n); totals[n-1]+=sum(pn.values())\n            for g,c in pn.items(): clipped[n-1]+=min(c,rn.get(g,0))\n    prec=[clipped[i]/totals[i] if totals[i] else 0.0 for i in range(order)]\n    if any(x<=0 for x in prec) or pl==0: return 0.0\n    bp=1.0 if pl>rl else math.exp(1.0-rl/pl)\n    return bp*math.exp(sum(math.log(x) for x in prec)/order)\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 1 predictions on your own held-out split.\')\n    ap.add_argument(\'--truth\', required=True, help=\'Your held-out reference JSON.\')\n    ap.add_argument(\'--prediction\', required=True, help=\'Your prediction JSON.\')\n    args=ap.parse_args()\n    truth=load_json(args.truth); pred=load_json(args.prediction)\n    validate(truth,\'truth\'); validate(pred,\'prediction\')\n    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")\n    ps=[pred[i][\'text\'].split() for i in truth]; rs=[truth[i][\'text\'].split() for i in truth]\n    pred_texts=[pred[i][\'text\'].strip() for i in truth]; ref_texts=[truth[i][\'text\'].strip() for i in truth]\n    try:\n        import torch\n        device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\n    except ImportError: device=None\n    _,_,f=bert_score.score(pred_texts, ref_texts, lang=\'en\', batch_size=8, verbose=False, **({\'device\':device} if device else {}))\n    bertscore=float(f.mean().item())\n    rouge=sum(rouge_l_f1(pred[i][\'text\'],truth[i][\'text\']) for i in truth)/len(truth)\n    b4=bleu4(ps,rs)\n    print(f\'Examples: {len(truth)}\')\n    print(f\'BERTScore F1: {bertscore:.6f}\')\n    print(f\'ROUGE-L F1:   {rouge:.6f}\')\n    print(f\'BLEU-4:       {b4:.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\', file=sys.stderr); sys.exit(1)\n'
(OUTPUT_DIR/'evaluate_phase1.py').write_bytes(EVALUATOR_SOURCE.encode())
assert hashlib.sha256((OUTPUT_DIR/'evaluate_phase1.py').read_bytes()).hexdigest() == '3f3a9e62be718b478e3f52b378fcc535350ab90b9cb84e3a4c49fec247abf1db'
del EVALUATOR_SOURCE
print('Original evaluator restored unchanged.')

## 5. Encoder–decoder model

| Component | Configuration |
|---|---|
| Encoder | 4 bidirectional self-attention blocks |
| Decoder | 4 causal self-attention + encoder cross-attention blocks |
| Width / heads | 512 / 8 |
| Feed-forward | SwiGLU, hidden width 1,024 |
| Vocabulary | Your fixed 50,000-token vocabulary |
| Embeddings | Shared between encoder, decoder and output projection |
| Positions / normalization | Fixed sinusoidal / RMSNorm |
| Parameters | **50,777,088** |

The encoder sees only the prefix. The decoder receives BOS followed by the shifted true continuation during training. It predicts continuation tokens using causal self-attention and cross-attention to the encoder. At inference, it starts from BOS and generates autoregressively, caching attention keys and values. No convolution or shared prefix-LM stack is used.

Training uses FP16 and two-process DDP. A GPU preflight chooses a fitting microbatch while keeping **65,536 corpus tokens per optimizer update**. The code below is the actual model implementation.


In [ ]:
MODEL_CODE = r'''"""LegalEncDec: randomly initialized encoder-decoder Transformer."""
import math
from dataclasses import dataclass
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

@dataclass
class ModelConfig:
    vocab_size: int = 50000
    d_model: int = 512
    n_heads: int = 8
    encoder_layers: int = 4
    decoder_layers: int = 4
    d_ff: int = 1024
    max_context: int = 2048
    checkpoint_blocks: bool = False
    loss_chunk: int = 512

class RMSNorm(nn.Module):
    def __init__(self, d):
        super().__init__(); self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        y = x.float() * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + 1e-6)
        return y.to(x.dtype) * self.weight

class Attention(nn.Module):
    def __init__(self, c):
        super().__init__(); self.heads = c.n_heads
        self.q = nn.Linear(c.d_model, c.d_model, bias=False)
        self.k = nn.Linear(c.d_model, c.d_model, bias=False)
        self.v = nn.Linear(c.d_model, c.d_model, bias=False)
        self.out = nn.Linear(c.d_model, c.d_model, bias=False)
    def split(self, x):
        b,t,d=x.shape
        return x.view(b,t,self.heads,d//self.heads).transpose(1,2)
    def forward(self, x, memory=None, causal=False, cache=None, use_cache=False):
        q=self.split(self.q(x)); cross=memory is not None
        if cross and cache is not None:
            k,v=cache
        else:
            kv=x if memory is None else memory
            k,v=self.split(self.k(kv)),self.split(self.v(kv))
            if cache is not None:
                k,v=torch.cat([cache[0],k],-2),torch.cat([cache[1],v],-2)
        # A cached single-token query can attend all saved keys.
        a=F.scaled_dot_product_attention(q,k,v,is_causal=causal and cache is None,dropout_p=0.0)
        a=a.transpose(1,2).contiguous().view_as(x)
        return self.out(a), (k,v) if use_cache else None

class FeedForward(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.up=nn.Linear(c.d_model,2*c.d_ff,bias=False)
        self.down=nn.Linear(c.d_ff,c.d_model,bias=False)
    def forward(self,x):
        g,v=self.up(x).chunk(2,-1)
        return self.down(F.silu(g)*v)

class EncoderBlock(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.norm1,self.norm2=RMSNorm(c.d_model),RMSNorm(c.d_model)
        self.attn,self.ff=Attention(c),FeedForward(c)
    def forward(self,x):
        x=x+self.attn(self.norm1(x))[0]
        return x+self.ff(self.norm2(x))

class DecoderBlock(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.norm1,self.norm2,self.norm3=[RMSNorm(c.d_model) for _ in range(3)]
        self.self_attn,self.cross_attn=Attention(c),Attention(c)
        self.ff=FeedForward(c)
    def forward(self,x,memory,cache=None,use_cache=False):
        a,s=self.self_attn(self.norm1(x),causal=True,cache=None if cache is None else cache[0],use_cache=use_cache)
        x=x+a
        a,c=self.cross_attn(self.norm2(x),memory=memory,cache=None if cache is None else cache[1],use_cache=use_cache)
        x=x+a
        return x+self.ff(self.norm3(x)),(s,c) if use_cache else None

class LegalEncDec(nn.Module):
    def __init__(self,c):
        super().__init__(); self.config=c
        assert c.d_model%c.n_heads==0 and c.d_model%2==0
        self.embedding=nn.Embedding(c.vocab_size,c.d_model)
        self.encoder=nn.ModuleList([EncoderBlock(c) for _ in range(c.encoder_layers)])
        self.decoder=nn.ModuleList([DecoderBlock(c) for _ in range(c.decoder_layers)])
        self.encoder_norm,self.decoder_norm=RMSNorm(c.d_model),RMSNorm(c.d_model)
        # Fixed positions; encoder and decoder use independent position indices.
        pos=torch.arange(c.max_context).float()[:,None]
        freq=torch.exp(torch.arange(0,c.d_model,2).float()*(-math.log(10000)/c.d_model))
        pe=torch.zeros(c.max_context,c.d_model)
        pe[:,0::2]=torch.sin(pos*freq);pe[:,1::2]=torch.cos(pos*freq)
        self.register_buffer('positions',pe,persistent=False)
        self.apply(self._init)
        for stack in [self.encoder,self.decoder]:
            for block in stack:
                nn.init.normal_(block.ff.down.weight,std=.02/math.sqrt(2*len(stack)))
    @staticmethod
    def _init(m):
        if isinstance(m,(nn.Linear,nn.Embedding)):nn.init.normal_(m.weight,std=.02)
    def embed(self,ids,offset=0):
        if ids.shape[1]+offset>self.config.max_context:raise ValueError('Context limit exceeded.')
        return self.embedding(ids)*math.sqrt(self.config.d_model)+self.positions[offset:offset+ids.shape[1]]
    def encode(self,source):
        x=self.embed(source)
        for block in self.encoder:
            x=checkpoint(block,x,use_reentrant=False) if self.training and self.config.checkpoint_blocks else block(x)
        return self.encoder_norm(x)
    def decode(self,decoder_ids,memory,past=None,use_cache=False):
        if past is not None and decoder_ids.shape[1]!=1:raise ValueError('Cached decoding needs one token.')
        offset=0 if past is None else past[0][0][0].shape[-2]
        x=self.embed(decoder_ids,offset); caches=[]
        for i,block in enumerate(self.decoder):
            if self.training and self.config.checkpoint_blocks and not use_cache:
                def run(a,b,layer=block):return layer(a,b)[0]
                x=checkpoint(run,x,memory,use_reentrant=False)
            else:
                x,cache=block(x,memory,None if past is None else past[i],use_cache)
                if use_cache:caches.append(cache)
        return self.decoder_norm(x),caches
    def forward(self,source,decoder_ids,labels=None):
        h,_=self.decode(decoder_ids,self.encode(source))
        if labels is None:return F.linear(h,self.embedding.weight)
        h=h.reshape(-1,self.config.d_model); y=labels.reshape(-1)
        total=h.new_zeros((),dtype=torch.float32)
        def ce(a,w,b):return F.cross_entropy(F.linear(a,w).float(),b,reduction='sum')
        for begin in range(0,len(y),self.config.loss_chunk):
            a,b=h[begin:begin+self.config.loss_chunk],y[begin:begin+self.config.loss_chunk]
            total+=checkpoint(ce,a,self.embedding.weight,b,use_reentrant=False) if self.training else ce(a,self.embedding.weight,b)
        return total/len(y)

# Each 1,024-token example consists of a prefix and its continuation.
# BOS is a decoder start token, not an extra counted corpus token.
def training_loss(model,rows,prefix):
    ids=rows[:,:-1]
    labels=ids[:,prefix:]
    decoder_ids=torch.cat([torch.full_like(labels[:,:1],2),labels[:,:-1]],1)
    return model(ids[:,:prefix],decoder_ids,labels)
'''
(OUTPUT_DIR/'model.py').write_text(MODEL_CODE)


## 6. Check both GPUs and fetch training-source metadata
This cell authenticates, pins the dataset revision, restores a compatible private checkpoint if one exists, and performs a small real-model GPU test. Temporary synthetic-data weights are discarded. The speed test prints a measurement; it does **not** reduce the 300M-token target.

In [ ]:
from pipeline import Pipeline

experiment = Pipeline(OUTPUT_DIR, settings)
experiment.setup()
config = json.loads((OUTPUT_DIR/'run_config.json').read_text())
print('Model parameters: 50,777,088')
print('Training target:', f"{config['train']['budget_tokens']:,}", 'input tokens')
print('Sources:', list(config['source_files']))

## 7. Fetch and preprocess data
Raw JSON is streamed from HF rather than loaded into a huge Python list. Sources receive token quotas and preparation time proportional to the **square root of their file sizes**. Final quotas are capped by available unique data; deficits are redistributed. There is no oversampling to fake 300M tokens.

Preprocessing applies Unicode NFC, whitespace/control-character cleanup, short/corrupt-text filtering and exact normalized-text deduplication. A content hash holds out whole documents before packing, so exact duplicates cannot cross splits. The original tokenizer is unchanged; its prior training provenance is unknown.

Training uses packed 1,024-token blocks in a fixed shuffled plan. BOS/EOS separate documents; attention may cross those boundaries. Up to eight genuine 512-word prompts and following 128-word targets per source are retained for validation. The 75-minute preparation limit means sampling covers the visited part of each source, **not a uniform sample of the entire file**. Near-duplicates are not removed.

In [ ]:
PREPROCESS_CODE = r'''
"""Streaming selection, disjoint validation, bounded reservoirs, and a fixed data plan."""
import argparse, hashlib, io, json, math, os, re, sqlite3, unicodedata, time
from collections import Counter
from pathlib import Path
import ijson
import numpy as np
import requests
from huggingface_hub import HfApi, hf_hub_url, hf_hub_download
from tokenizers import Tokenizer

SOURCES = ['acts','high_courts','supreme_court','tribunals']

def atomic_json(path,obj):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp')
    with tmp.open('w',encoding='utf8') as f:
        json.dump(obj,f,ensure_ascii=False,indent=2); f.flush(); os.fsync(f.fileno())
    os.replace(tmp,path)

def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(8<<20),b''): h.update(b)
    return h.hexdigest()

def clean_text(text):
    text=unicodedata.normalize('NFC',text)
    text=re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]',' ',text)
    return ' '.join(text.split())

def get_text(rec):
    if isinstance(rec,str): return rec
    if isinstance(rec,list) and all(isinstance(x,str) for x in rec): return '\n'.join(rec)
    if isinstance(rec,dict):
        for key in ['text','judgment','judgement','content','document','raw_text','full_text']:
            if key in rec:
                v=rec[key]
                if isinstance(v,str): return v
                if isinstance(v,list) and all(isinstance(x,str) for x in v): return '\n'.join(v)
    raise ValueError('Unrecognized text record. Expected a string or a text/content/judgment field. '
                     f'Record type={type(rec).__name__}; keys={list(rec)[:15] if isinstance(rec,dict) else "n/a"}. '
                     'Inspect one record and update get_text explicitly; do not silently train on metadata.')

def records(stream):
    stream=io.BufferedReader(stream,buffer_size=1<<20)
    beginning=stream.peek(4096).lstrip()
    if beginning.startswith(b'['):
        yield from ijson.items(stream,'item')
        return
    if not beginning.startswith(b'{'): raise ValueError('Expected JSON array/object or JSONL.')
    m=re.match(rb'\{\s*"([^"\\]+)"\s*:',beginning)
    if not m: raise ValueError('Cannot recognize JSON schema from its first key.')
    key=m.group(1).decode()
    if key in ['text','judgment','judgement','content','document','raw_text','full_text','id']:
        # Root records (including JSONL), not a giant ID-to-document mapping.
        for item in ijson.items(stream,'',multiple_values=True): yield item
    elif key in ['data','documents','train']:
        yield from ijson.items(stream,key+'.item')
    else:
        for _,value in ijson.kvitems(stream,''): yield value

class Reservoir:
    def __init__(self,path,capacity,width,seed):
        self.path=Path(path); self.capacity=capacity; self.width=width
        self.seen=0; self.rng=np.random.default_rng(seed)
        self.array=np.memmap(path,mode='w+',dtype='<u2',shape=(capacity,width))
    def add(self,row):
        self.seen+=1
        j=self.seen-1 if self.seen<=self.capacity else int(self.rng.integers(self.seen))
        if j<self.capacity: self.array[j]=row
    def close(self):
        used=min(self.seen,self.capacity); self.array.flush(); del self.array
        with self.path.open('r+b') as f: f.truncate(used*self.width*2)
        return used

class Packer:
    def __init__(self,reservoir,length): self.r=reservoir; self.length=length; self.buf=[]
    def add(self,ids):
        self.buf.extend(ids)
        end=0
        while len(self.buf)-end>=self.length+1:
            self.r.add(self.buf[end:end+self.length+1]); end+=self.length
        if end: self.buf=self.buf[end:]

# Capped proportional allocation: no source is sampled more than its available pool.
def allocate(total,capacities,weights):
    cap=np.asarray(capacities,dtype=np.int64)
    w=np.asarray(weights,dtype=np.float64)
    if total>int(cap.sum()):
        raise ValueError(f'Only {int(cap.sum()):,} unique blocks available for {total:,} requested. '
                         'Increase selection_multiplier and rebuild data; no silent repeats.')
    out=np.zeros(len(cap),dtype=np.int64)
    while out.sum()<total:
        room=cap-out; active=room>0; rem=total-int(out.sum())
        probs=np.where(active,w,0); probs=probs/probs.sum()
        add=np.minimum(room,np.floor(rem*probs).astype(np.int64))
        if not add.any():
            candidates=np.flatnonzero(active)
            pick=candidates[np.argmax((rem*probs)[candidates])]; add[pick]=1
        out+=add
    return out

def fetch_source(cfg,name,out,tokenizer,db):
    root=Path(cfg['root']); data_cfg=cfg['data']; L=cfg['train']['seq_len']
    source_no=SOURCES.index(name); info=cfg['source_files'][name]
    marker=out/(name+'.stats.json'); trainpath=out/(name+'.pool.bin')
    if marker.exists():
        result=json.loads(marker.read_text())
        if not trainpath.exists(): raise RuntimeError('Incomplete preparation; restore data or use a fresh root.')
        return result
    db.execute('DELETE FROM seen WHERE source=?',(name,)); db.commit()
    seed=cfg['seed']+source_no*1009
    weights=np.sqrt([cfg['source_files'][s]['size'] for s in SOURCES]); weights/=weights.sum()
    cap=max(64,math.ceil(cfg['train']['total_samples']*weights[source_no]*1.10))
    train=Reservoir(trainpath,cap,L+1,seed)
    val=Reservoir(out/(name+'.val.bin'),data_cfg['val_blocks_per_source'],L+1,seed+1)
    trainpack,valpack=Packer(train,L),Packer(val,L)
    pair_rng=np.random.default_rng(seed+2); pairs=[]; pair_seen=0
    stats=Counter()
    deadline=min(time.time()+data_cfg['source_seconds'].get(name,900),cfg.get('preparation_deadline_epoch',float('inf'))-45)
    print(f'{name}: time-bounded scan, capacity {cap:,} blocks',flush=True)
    url=hf_hub_url(cfg['dataset_repo'],info['path'],repo_type='dataset',revision=cfg['dataset_revision'])
    with requests.get(url,headers={'Authorization':'Bearer '+os.environ['HF_TOKEN']},
                      stream=True,timeout=(20,30)) as response:
        response.raise_for_status(); response.raw.decode_content=True
        for rec in records(response.raw):
            if time.time()>=deadline:
                stats['time_limited']=1; break
            if train.seen>=cap and val.seen>0 and len(pairs)>=data_cfg['dev_pairs_per_source']:
                stats['quota_limited']=1; break
            stats['read_documents']+=1
            text=clean_text(get_text(rec)); words=text.split()
            if len(words)<data_cfg['min_words'] or text.count('\ufffd')>0.005*max(1,len(text)):
                stats['quality_rejected']+=1; continue
            if sum(ch.isalpha() for ch in text)/max(1,len(text))<0.3:
                stats['quality_rejected']+=1; continue
            canonical=text.casefold(); digest=hashlib.sha256(canonical.encode()).digest()
            heldout=int.from_bytes(digest[:8],'little')%1000 < data_cfg['holdout_per_mille']
            stats['eligible_characters']+=len(text)
            result=db.execute('INSERT OR IGNORE INTO seen VALUES (?,?)',(digest,name))
            if result.rowcount==0: stats['duplicates_rejected']+=1; continue
            if heldout:
                stats['heldout_documents']+=1
                # One contiguous 512-word prompt and its true 128-word continuation.
                if len(words)>=640:
                    begin=int.from_bytes(digest[16:24],'little')%(len(words)-640+1)
                    pair={'id':name+'_'+digest.hex()[:24], 'source':name,
                          'prompt':' '.join(words[begin:begin+512]),
                          'truth':' '.join(words[begin+512:begin+640])}
                    pair_seen+=1
                    j=pair_seen-1 if pair_seen<=data_cfg['dev_pairs_per_source'] else int(pair_rng.integers(pair_seen))
                    if j<data_cfg['dev_pairs_per_source']:
                        if j==len(pairs): pairs.append(pair)
                        else: pairs[j]=pair
                ids=tokenizer.encode(text,add_special_tokens=False).ids
                valpack.add([2]+ids+[3])
            else:
                stats['selected_documents']+=1
                ids=tokenizer.encode(text,add_special_tokens=False).ids
                if not ids: continue
                if max(ids)>=50000 or min(ids)<0: raise ValueError('Tokenizer IDs outside vocabulary.')
                stats['selected_tokens']+=len(ids)+2
                trainpack.add([2]+ids+[3])
            if stats['read_documents']%5000==0:
                db.commit()
                print(name,{k:stats[k] for k in ['read_documents','selected_documents','selected_tokens']},flush=True)
    db.commit()
    stats['pool_rows']=train.close(); stats['candidate_blocks']=train.seen
    stats['val_rows']=val.close(); stats['dev_pairs']=len(pairs)
    if stats['pool_rows']==0:
        raise RuntimeError(f'{name}: no usable blocks within time allowance; inspect access, schema and filters.')
    atomic_json(out/(name+'.pairs.json'),pairs)
    atomic_json(marker,dict(stats))
    return dict(stats)

def prepare(cfg):
    root=Path(cfg['root']); out=root/'data'; out.mkdir(exist_ok=True)
    complete=out/'manifest.json'
    if complete.exists():
        m=json.loads(complete.read_text())
        assert m['signature']==cfg['resume_signature'], 'Prepared data belongs to a different configuration.'
        for name,meta in m['files'].items():
            p=out/name
            assert p.exists() and p.stat().st_size==meta['bytes'],f'Missing/truncated prepared file: {name}'
        print('Reusing completed corpus:',m['training_tokens'],'tokens',flush=True); return
    # Preparation is atomic per source. An interrupted source is rescanned from its beginning.
    tokenizer=Tokenizer.from_file(str(root/'tokenizer.json'))
    db=sqlite3.connect(out/'dedup.sqlite'); db.execute('PRAGMA journal_mode=WAL')
    db.execute('CREATE TABLE IF NOT EXISTS seen (digest BLOB PRIMARY KEY,source TEXT) WITHOUT ROWID')
    stats={s:fetch_source(cfg,s,out,tokenizer,db) for s in SOURCES}; db.close()
    available=sum(stats[s]['pool_rows'] for s in SOURCES)
    per=cfg['train']['samples_per_update']
    wanted=min(cfg['train']['total_samples'],available)//per*per
    if wanted<per: raise RuntimeError('Not enough unique data for one update.')
    counts=allocate(wanted,[stats[s]['pool_rows'] for s in SOURCES],
                    [cfg['source_files'][s]['size']**0.5 for s in SOURCES])
    L=cfg['train']['seq_len']; rng=np.random.default_rng(cfg['seed']+123)
    plan=[]; pairs=[]; val_arrays=[]; files={}
    # Keep source reservoirs until the final manifest commits, making finalization restartable.
    for si,s in enumerate(SOURCES):
        n=int(counts[si]); source=np.memmap(out/(s+'.pool.bin'),dtype='<u2',mode='r',shape=(stats[s]['pool_rows'],L+1))
        chosen=rng.permutation(len(source))[:n]
        dest=out/(s+'.train.bin')
        with dest.open('wb') as f:
            for offset in range(0,n,2048): source[chosen[offset:offset+2048]].tofile(f)
        del source
        plan.extend((si,i) for i in range(n))
        pairs.extend(json.loads((out/(s+'.pairs.json')).read_text()))
        val_arrays.append(np.fromfile(out/(s+'.val.bin'),dtype='<u2').reshape(-1,L+1))
        stats[s]['training_rows']=n; stats[s]['training_tokens']=n*L
        files[dest.name]={'bytes':dest.stat().st_size,'sha256':sha_file(dest)}
    plan=np.asarray(plan,dtype=np.uint32); rng.shuffle(plan)
    np.save(out/'plan.npy',plan,allow_pickle=False)
    validation=np.concatenate(val_arrays)
    if len(validation)<4 or len(pairs)<4: raise RuntimeError('Too little held-out data within preparation allowance.')
    np.save(out/'validation.npy',validation,allow_pickle=False)
    atomic_json(out/'dev_prompts.json',{p['id']:{'text':p['prompt']} for p in pairs})
    atomic_json(out/'dev_truth.json',{p['id']:{'text':p['truth']} for p in pairs})
    for name in ['plan.npy','validation.npy','dev_prompts.json','dev_truth.json']:
        p=out/name; files[name]={'bytes':p.stat().st_size,'sha256':sha_file(p)}
    if len(plan)*L < cfg['train']['budget_tokens']:
        print(f'DATA SHORTFALL: {len(plan)*L:,} unique tokens prepared versus the fixed {cfg["train"]["budget_tokens"]:,}-token target. No repetitions added.',flush=True)
    atomic_json(complete,{'signature':cfg['resume_signature'],'source_stats':stats,
                          'training_tokens':len(plan)*L,'files':files})
    for s in SOURCES: (out/(s+'.pool.bin')).unlink(missing_ok=True)
    for pattern in ['dedup.sqlite*','*.val.bin','*.pairs.json']:
        for p in out.glob(pattern): p.unlink(missing_ok=True)
    print('Final source token counts:',{s:stats[s]['training_tokens'] for s in SOURCES},flush=True)
    print('Total unique selected input tokens:',len(plan)*L,flush=True)

class PlannedData:
    def __init__(self,root,seq_len):
        self.root=Path(root)/'data'; self.seq_len=seq_len
        self.plan=np.load(self.root/'plan.npy',mmap_mode='r')
        self.arrays=[np.memmap(self.root/(s+'.train.bin'),mode='r',dtype='<u2').reshape(-1,seq_len+1) if (self.root/(s+'.train.bin')).stat().st_size else np.empty((0,seq_len+1),dtype=np.uint16) for s in SOURCES]
    def batch(self,start,count):
        rows=self.plan[start:start+count]
        if len(rows)!=count: raise IndexError('Training plan exhausted; refusing to cycle/repeat it.')
        return np.stack([self.arrays[int(s)][int(i)] for s,i in rows]).astype(np.int64)

if __name__=='__main__':
    ap=argparse.ArgumentParser(); ap.add_argument('--config',required=True); args=ap.parse_args()
    prepare(json.loads(Path(args.config).read_text()))
'''
(OUTPUT_DIR/'data_pipeline.py').write_text(PREPROCESS_CODE)


In [ ]:
experiment.prepare_data()
manifest = json.loads((OUTPUT_DIR/'data/manifest.json').read_text())
print('Prepared unique input tokens:', f"{manifest['training_tokens']:,}")

## 8. Train or resume on both GPUs
`train()` launches two DDP workers automatically; no terminal commands are needed. Model, optimizer, scaler, sample cursor and per-rank RNG states are saved every 100 updates or about five minutes. The LR schedule continues from its saved step. The clock and original data plan do not reset when you rerun a cell.

Training ends at the target, data exhaustion, or its deadline. Remote backups are bounded and optional; a failed upload is reported. A hard kill restores the last committed checkpoint, not the exact instruction where the process stopped. A VM reset requires a successfully uploaded private backup. Prepared-data progress resumes at completed-source boundaries.

In [ ]:
experiment.train()

## 9. Training and validation loss
Both curves measure continuation cross-entropy: 768 encoder tokens condition 256 decoder targets. Validation uses held-out documents. Perplexity is conditional continuation perplexity, not BERTScore.


In [ ]:
import math
import matplotlib.pyplot as plt

def read_jsonl(name):
    path = OUTPUT_DIR/name
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()] if path.exists() else []

train_log = read_jsonl('metrics.jsonl')
val_log = read_jsonl('validation_metrics.jsonl')
if train_log:
    plt.figure(figsize=(9, 4))
    plt.plot([x['step'] for x in train_log], [x['loss'] for x in train_log], label='Training loss')
    if val_log:
        plt.plot([x['step'] for x in val_log], [x['prefix_validation_loss'] for x in val_log], label='Continuation validation loss')
        print('Latest continuation validation perplexity:', round(val_log[-1]['prefix_perplexity'], 3))
    plt.xlabel('Optimizer step'); plt.ylabel('Loss'); plt.legend(); plt.grid(alpha=.25)
    plt.tight_layout(); plt.savefig(OUTPUT_DIR/'loss.png', dpi=120); plt.show()
else:
    print('No training measurements have been recorded yet.')

## 10. Evaluate using your exact Phase 1 script
The best validation-loss checkpoint is used, with latest as fallback. Generation produces 128 words for each held-out 512-word prompt, saving progress after each example. The original evaluator reports BERTScore F1, ROUGE-L and BLEU. Its pretrained scorer is used only for measurement.

This local evaluation has up to **32 examples**, drawn from the held-out documents. It is a small validation check, not an official test result or a promised F1. Its examples can differ from the 90M run because selection ends at a different data budget; compare models on identical examples before claiming an improvement. The validation split also selects the best checkpoint.

No submission files or official test-set pipeline are included. An interrupted or incomplete evaluation is reported as such.


In [ ]:
evaluation = experiment.evaluate()
print(json.dumps(evaluation, indent=2))

## 11. Save and print the final result

In [ ]:
experiment.finish()
print('Recover training from:', OUTPUT_DIR/'latest.pt')
print('Best validation model:', OUTPUT_DIR/'best.pt')
print('Training report:', OUTPUT_DIR/'run_report.json')
print('Evaluation report:', OUTPUT_DIR/'evaluation_report.json')

**Resume:** use this notebook and the same HF account/private repository, then Run All. `latest.pt` contains model, optimizer, scaler, completed update, data cursor and per-rank RNG states. `best.pt` contains inference weights and model configuration. Recovery is to the last committed checkpoint. New sessions get a new 10-hour clock; multiple sessions are not counted as one 10-hour run.

**Verified locally:** CPU model checks passed, including cached decoding and loss gradients. A small interrupted training run resumed to exactly the same weights as an uninterrupted run, saving both checkpoint types.

**Checks:** the notebook runs parameter-count, causal-mask, cached-decoding, loss/gradient and token-accounting checks before training, then a real GPU preflight. This deliverable has not been trained on Kaggle; it makes no measured runtime or F1 claim.

Reference: [PyTorch scaled dot-product attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html).
